# LAIQMS Capstone: LLM-Grounded Alpha Research and Risk System

This is the full bootcamp capstone in a natural financial-model-building order:

1. Quant data, returns, labels, options, and baseline risk.
2. Feature engine using data structures and system-design discipline.
3. Walk-forward ML/AI model with honest validation.
4. LLM/RAG research layer grounded in retrieved evidence and market data.
5. Portfolio construction, execution costs, option overlay, and risk report.
6. AWS/system design package with security, monitoring, cost, and final-defense prompts.

This is the capstone project from Columbia MAFN's LAIQMS bootcamp — a program preparing incoming students for AI/ML/LLM/quant/systems-design interviews (see [README](../README.md) for context on the program).


## LAIQMS Coverage Map

| Module | LAIQMS | Main subsections |
|---|---|---|
| S1 Data, Returns, Labels | Q, M | Calculus, Complex, Linear Algebra, Probability, Stochastic, Instruments, Portfolio, Black-Scholes, Monte Carlo, Time-Series Validation |
| S2 Feature Engine + DSA | S, Q | Arrays, Heaps, Monotonic Stacks, Binary Search, Graphs, Dynamic Programming, System Design Fundamentals, Portfolio Risk |
| S3 ML + AI Validation | M, AI, Q | Supervised Learning, Trees/Ensembles, Metrics, Time-Series CV, Neural Networks, Calibration, Probability |
| S4 LLM Research Layer | L, AI, Q | Tokenization, Embeddings, Softmax, Attention, RAG, Calibration, Finance Grounding |
| S5 Portfolio, Execution, Risk | Q, M, S | Portfolio Theory, Instruments, Monte Carlo, Model Metrics, Stream/Queue Controls, Trading System Design |
| S6 AWS + Final Defense | S, AWS, L, AI, Q, M | AWS Cloud, IAM, Compute, Storage, Networking, Monitoring, Billing, Applied ML/Trading System Design |


# S1 - Data, Returns, Labels

Build the market data contract, leak-free return labels, PCA risk model, option-pricing layer, and baseline portfolio math.


## Mapped LAIQMS Subsections

- Q: Calculus & Differential Equations
- Q: Complex Numbers
- Q: Linear Algebra
- Q: Probability & Statistics
- Q: Stochastic Processes
- Q: Financial Instruments & Derivatives
- Q: Portfolio Theory, Factors & Risk
- Q: Black-Scholes
- Q: Monte Carlo & Numerical Methods
- M: Time-Series Validation & Leakage Control


## S1 - Data, Returns, Labels

**Natural financial-model order:** before ML, LLMs, or AWS, the system needs a clean market data contract.

**LAIQMS mapping**
- Q / Calculus: log returns as discrete derivatives of log price.
- Q / Complex Numbers: FFT identifies cyclic structure in returns.
- Q / Linear Algebra: covariance, eigenvalues, PCA, Cholesky.
- Q / Probability & Statistics: moments, quantiles, VaR, expected shortfall.
- Q / Stochastic Processes: regime-switching GBM-like simulation.
- Q / Financial Instruments: equity, option metadata, risk-free curve proxy.
- Q / Portfolio Theory: covariance shrinkage, beta, Sharpe, risk budget.
- Q / Black-Scholes: option price and Greeks.
- Q / Monte Carlo: simulated terminal distributions and option checks.
- M / Time-Series Validation: point-in-time labels only use future returns after the feature date.


In [1]:
import json
import math
import os
import heapq
import bisect
from collections import deque, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import brentq
from sklearn.decomposition import PCA

RNG_SEED = 113353
rng = np.random.default_rng(RNG_SEED)

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

TICKERS = ["AAPL", "MSFT", "JPM", "GS", "XOM", "UNH", "CAT", "GE", "IBM", "DIS", "KO", "WMT"]
N_DAYS = 756
DATES = pd.bdate_range("2023-01-03", periods=N_DAYS)
RISK_FREE = 0.043

def annualize_return(daily_mean):
    return (1.0 + daily_mean) ** 252 - 1.0

def annualize_vol(daily_std):
    return daily_std * np.sqrt(252)

def max_drawdown(series):
    wealth = pd.Series(series).astype(float)
    peak = wealth.cummax()
    dd = wealth / peak - 1.0
    return float(dd.min())

def black_scholes_call(S, K, T, r, sigma):
    sigma = max(float(sigma), 1e-8)
    T = max(float(T), 1e-8)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T)
    theta = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    return {"price": float(price), "delta": float(delta), "gamma": float(gamma), "vega": float(vega), "theta": float(theta)}

def implied_vol_call(price, S, K, T, r):
    def f(sig):
        return black_scholes_call(S, K, T, r, sig)["price"] - price
    try:
        return float(brentq(f, 1e-4, 4.0, maxiter=100))
    except ValueError:
        return np.nan

print("Setup complete. Seed:", RNG_SEED)


Setup complete. Seed: 113353


In [2]:
# Simulate an institutional-grade equity universe with latent macro/sector/liquidity/volatility factors.
n = len(TICKERS)
k = 5
factor_names = ["market", "rates", "energy", "quality", "liquidity"]

regime = np.zeros(N_DAYS, dtype=int)
p_stay = {0: 0.965, 1: 0.92, 2: 0.88}
for t in range(1, N_DAYS):
    if rng.random() < p_stay[regime[t - 1]]:
        regime[t] = regime[t - 1]
    else:
        regime[t] = rng.choice([0, 1, 2], p=[0.72, 0.18, 0.10])

regime_mu = np.array([0.00035, -0.00015, -0.00055])
regime_vol = np.array([0.0085, 0.014, 0.024])
factor_cov = np.array([
    [1.00, -0.22, 0.15, 0.30, -0.35],
    [-0.22, 1.00, -0.08, -0.10, 0.25],
    [0.15, -0.08, 1.00, 0.04, -0.10],
    [0.30, -0.10, 0.04, 1.00, -0.28],
    [-0.35, 0.25, -0.10, -0.28, 1.00],
])
L = np.linalg.cholesky(factor_cov)
raw = rng.standard_normal((N_DAYS, k)) @ L.T
factor_rets = pd.DataFrame(raw * regime_vol[regime, None] + regime_mu[regime, None] / k, index=DATES, columns=factor_names)

exposures = pd.DataFrame(rng.normal(0.0, 0.55, size=(n, k)), index=TICKERS, columns=factor_names)
exposures["market"] = rng.normal(1.0, 0.20, n)
exposures["quality"] = rng.normal(0.25, 0.45, n)
idio = rng.normal(0, 0.0065, size=(N_DAYS, n))
drift = rng.normal(0.00018, 0.00008, n)
returns = factor_rets.values @ exposures.T.values + drift + idio
returns = pd.DataFrame(returns, index=DATES, columns=TICKERS).clip(-0.12, 0.12)
prices = 100 * np.exp(returns.cumsum())

volumes = pd.DataFrame(
    rng.lognormal(mean=14.4, sigma=0.33, size=(N_DAYS, n)) * (1.0 + np.abs(returns.values) * 18),
    index=DATES,
    columns=TICKERS,
)
labels = (returns.shift(-5).rolling(5).sum().shift(-4) > returns.shift(-5).rolling(5).sum().shift(-4).median(axis=1).values[:, None]).astype(int)
labels = labels.iloc[:-10]

DATA = {"prices": prices, "returns": returns, "volumes": volumes, "factors": factor_rets, "exposures": exposures, "regime": pd.Series(regime, index=DATES, name="regime")}
print(prices.tail(2).round(2))
print("Returns shape:", returns.shape, "Labels shape:", labels.shape)


              AAPL   MSFT    JPM      GS    XOM    UNH    CAT     GE    IBM  \
2025-11-24  106.63  72.93  57.25  107.99  53.87  71.57  47.51  62.36  61.27   
2025-11-25  106.67  73.66  57.40  108.86  53.06  70.90  48.94  63.00  61.48   

              DIS     KO    WMT  
2025-11-24  60.84  70.85  37.72  
2025-11-25  62.11  70.25  38.95  
Returns shape: (756, 12) Labels shape: (746, 12)


In [3]:
# Quant audit: moments, PCA, FFT, risk, portfolio baseline, and option pricing.
ret = DATA["returns"].dropna()
mu_daily = ret.mean()
cov_daily = ret.cov()
cov_ann = cov_daily * 252
vol_ann = ret.std() * np.sqrt(252)
corr = ret.corr()

eigvals, eigvecs = np.linalg.eigh(cov_ann.values)
order = eigvals.argsort()[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]
pca = PCA(n_components=4, random_state=RNG_SEED).fit(ret.fillna(0.0))

# Complex numbers: FFT on market return proxy.
market_proxy = ret.mean(axis=1).values
fft = np.fft.rfft(market_proxy - market_proxy.mean())
freqs = np.fft.rfftfreq(len(market_proxy), d=1)
dominant = int(np.argmax(np.abs(fft[1:])) + 1)
cycle_days = float(1.0 / freqs[dominant]) if freqs[dominant] else np.inf

# Mean-variance baseline with ridge-stabilized inverse covariance.
mu_ann = mu_daily * 252
shrink = 0.12
diag = np.diag(np.diag(cov_ann.values))
sigma_shrunk = (1 - shrink) * cov_ann.values + shrink * diag
inv_sigma = np.linalg.pinv(sigma_shrunk)
raw_w = inv_sigma @ (mu_ann.values - RISK_FREE)
w = raw_w / np.sum(np.abs(raw_w))
weights = pd.Series(w, index=TICKERS, name="baseline_weight")

port_ret = (ret @ weights).rename("baseline_portfolio")
wealth = (1 + port_ret).cumprod()
var95 = float(np.quantile(port_ret, 0.05))
es95 = float(port_ret[port_ret <= var95].mean())
sharpe = float((port_ret.mean() * 252 - RISK_FREE) / (port_ret.std() * np.sqrt(252)))

S0 = float(prices.iloc[-1]["AAPL"])
sigma_aapl = float(vol_ann["AAPL"])
opt = black_scholes_call(S0, S0 * 1.02, 30 / 365, RISK_FREE, sigma_aapl)
iv_check = implied_vol_call(opt["price"], S0, S0 * 1.02, 30 / 365, RISK_FREE)

# Monte Carlo sanity check for the same option.
z = rng.standard_normal(25000)
ST = S0 * np.exp((RISK_FREE - 0.5 * sigma_aapl**2) * (30 / 365) + sigma_aapl * np.sqrt(30 / 365) * z)
mc_price = float(np.exp(-RISK_FREE * (30 / 365)) * np.maximum(ST - S0 * 1.02, 0).mean())

s1_report = {
    "laiqms_modules": ["Q", "M"],
    "dominant_fft_cycle_days": cycle_days,
    "pca_explained_variance": pca.explained_variance_ratio_.round(4).tolist(),
    "portfolio_sharpe": sharpe,
    "daily_var_95": var95,
    "daily_expected_shortfall_95": es95,
    "max_drawdown": max_drawdown(wealth),
    "bs_call": opt,
    "implied_vol_check": iv_check,
    "mc_call_price": mc_price,
    "weights": weights.round(4).to_dict(),
}
(ARTIFACTS / "s1_quant_data_contract.json").write_text(json.dumps(s1_report, indent=2))
print(json.dumps({k: s1_report[k] for k in ["portfolio_sharpe", "daily_var_95", "mc_call_price"]}, indent=2))


{
  "portfolio_sharpe": 0.718251102201304,
  "daily_var_95": -0.005410259217828249,
  "mc_call_price": 2.2440891218373586
}


# S2 - Feature Engine + Data Structures

Convert quant concepts into point-in-time model features using interview-grade DSA patterns and a pipeline DAG.


## Mapped LAIQMS Subsections

- S: Arrays
- S: Strings
- S: Heaps and Priority Queues
- S: Monotonic Stacks
- S: Binary Search
- S: Graphs
- S: Dynamic Programming
- S: System Design Fundamentals & Estimation
- Q: Portfolio Theory, Factors & Risk


## S2 - Feature Engine + Data Structures

**Natural order:** once the quant data contract exists, build features. This is where coding-interview patterns become finance infrastructure.

**LAIQMS mapping**
- S / Arrays: vectorized returns, rolling windows, feature matrices.
- S / Heaps: top-k momentum candidates.
- S / Monotonic Stacks: rolling drawdown and maximum tracking.
- S / Binary Search: locate event dates and rebalance boundaries.
- S / Graphs: pipeline dependency DAG and topological order.
- S / Dynamic Programming: transaction-cost-aware position smoothing.
- Q / Portfolio Risk: features remain tied to volatility, beta, correlation, and factors.


In [4]:
def rolling_zscore(df, window):
    mean = df.rolling(window).mean()
    std = df.rolling(window).std().replace(0, np.nan)
    return (df - mean) / std

def deque_rolling_mean(values, window):
    q, total, out = deque(), 0.0, []
    for x in values:
        q.append(float(x)); total += float(x)
        if len(q) > window:
            total -= q.popleft()
        out.append(total / len(q))
    return np.array(out)

def rolling_max_drawdown(values, window=63):
    out = []
    for i in range(len(values)):
        w = np.asarray(values[max(0, i - window + 1): i + 1], dtype=float)
        if len(w) < 2:
            out.append(0.0)
        else:
            out.append(max_drawdown(pd.Series(w / w[0])))
    return np.array(out)

def top_k_momentum(momentum_row, k=4):
    heap = []
    for ticker, val in momentum_row.dropna().items():
        heapq.heappush(heap, (float(val), ticker))
        if len(heap) > k:
            heapq.heappop(heap)
    return sorted([(t, v) for v, t in heap], key=lambda x: -x[1])

def topo_sort(graph):
    indeg = defaultdict(int)
    for node, deps in graph.items():
        indeg[node] += 0
        for dep in deps:
            indeg[dep] += 0
            indeg[node] += 1
    q = deque([n for n, d in indeg.items() if d == 0])
    order = []
    while q:
        n = q.popleft()
        order.append(n)
        for child, deps in graph.items():
            if n in deps:
                indeg[child] -= 1
                if indeg[child] == 0:
                    q.append(child)
    if len(order) != len(indeg):
        raise ValueError("Cycle in feature DAG")
    return order


In [5]:
ret = DATA["returns"]
prices = DATA["prices"]
volumes = DATA["volumes"]

mom_5 = prices.pct_change(5)
mom_21 = prices.pct_change(21)
vol_21 = ret.rolling(21).std() * np.sqrt(252)
vol_z = rolling_zscore(vol_21, 63)
dollar_vol_z = rolling_zscore(prices * volumes, 63)

beta_63 = pd.DataFrame(index=ret.index, columns=TICKERS, dtype=float)
market = ret.mean(axis=1)
for t in TICKERS:
    cov = ret[t].rolling(63).cov(market)
    var = market.rolling(63).var()
    beta_63[t] = cov / var

drawdown_63 = pd.DataFrame({t: rolling_max_drawdown(prices[t].values, 63) for t in TICKERS}, index=prices.index)
top_momentum_last = top_k_momentum(mom_21.iloc[-1], k=4)
rebalance_dates = list(prices.index[::21])
locate_idx = bisect.bisect_left(rebalance_dates, prices.index[250])

feature_panels = {
    "mom_5": mom_5,
    "mom_21": mom_21,
    "vol_21": vol_21,
    "vol_z": vol_z,
    "dollar_vol_z": dollar_vol_z,
    "beta_63": beta_63,
    "drawdown_63": drawdown_63,
}
rows = []
for name, panel in feature_panels.items():
    tmp = panel.stack().rename(name).reset_index()
    tmp.columns = ["date", "ticker", name]
    rows.append(tmp)
features = rows[0]
for r in rows[1:]:
    features = features.merge(r, on=["date", "ticker"], how="outer")

# Join the leak-free label from S1.
labels_long = labels.stack().rename("target").reset_index()
labels_long.columns = ["date", "ticker", "target"]
model_table = features.merge(labels_long, on=["date", "ticker"], how="inner").dropna()

FEATURE_COLUMNS = ["mom_5", "mom_21", "vol_21", "vol_z", "dollar_vol_z", "beta_63", "drawdown_63"]
PIPELINE_DAG = {
    "raw_prices": [],
    "returns": ["raw_prices"],
    "labels": ["returns"],
    "rolling_features": ["returns", "raw_prices"],
    "risk_features": ["returns"],
    "model_table": ["labels", "rolling_features", "risk_features"],
    "walk_forward_model": ["model_table"],
    "portfolio": ["walk_forward_model", "risk_features"],
}
dag_order = topo_sort(PIPELINE_DAG)

S2 = {
    "laiqms_modules": ["S", "Q"],
    "feature_columns": FEATURE_COLUMNS,
    "model_table_rows": int(len(model_table)),
    "top_momentum_last": top_momentum_last,
    "rebalance_date_example_index": int(locate_idx),
    "pipeline_topological_order": dag_order,
    "complexities": {
        "rolling_features_vectorized": "roughly O(T*N) per feature",
        "top_k_heap": "O(N log k)",
        "date_binary_search": "O(log T)",
        "dag_topological_sort": "O(V+E)",
    },
}
(ARTIFACTS / "s2_feature_catalog.json").write_text(json.dumps(S2, indent=2))
print(model_table.tail(3).round(4))
print(json.dumps({k: S2[k] for k in ["model_table_rows", "pipeline_topological_order"]}, indent=2))


           date ticker   mom_5  mom_21  vol_21   vol_z  dollar_vol_z  beta_63  \
8889 2025-11-11    DIS -0.0300 -0.0279  0.2034 -0.9210       -1.6383   1.2338   
8890 2025-11-11     KO  0.0160  0.0145  0.1602 -0.9596       -0.7252   0.6859   
8891 2025-11-11    WMT -0.0087  0.0453  0.3461 -0.9134        0.5449   1.6868   

      drawdown_63  target  
8889      -0.1977       1  
8890      -0.1027       1  
8891      -0.2398       0  
{
  "model_table_rows": 7968,
  "pipeline_topological_order": [
    "raw_prices",
    "returns",
    "labels",
    "rolling_features",
    "risk_features",
    "model_table",
    "walk_forward_model",
    "portfolio"
  ]
}


# S3 - ML + AI Model Validation

Train an honest walk-forward ensemble and produce a model card with leakage, calibration, and risk caveats.


## Mapped LAIQMS Subsections

- M: Supervised Learning I - Bias, Variance & Splits
- M: Supervised Learning II - Models & Regularization
- M: Trees, Ensembles & Feature Selection
- M: Metrics, Model Selection & Imbalanced Data
- M: Time-Series Validation & Leakage Control
- AI: Neural Network Foundations
- AI: Applied AI Evaluation & Calibration
- AI: AI Systems, Fine-Tuning & Deployment
- Q: Probability & Statistics


## S3 - ML + AI Model Validation

**Natural order:** after features, train the model. The goal is not fake accuracy; it is leak-free, defensible validation.

**LAIQMS mapping**
- M / Bias-Variance: compare linear and nonlinear models.
- M / Models & Regularization: logistic regression, random forest, gradient boosting, MLP.
- M / Time-Series Validation: expanding walk-forward split.
- AI / Neural Networks: small MLP classifier.
- AI / Calibration: Brier score, probability bins, thresholds.
- Q / Probability: expected value of a probabilistic signal.


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, precision_recall_fscore_support, accuracy_score

def expanding_splits(unique_dates, train_min=250, test_size=42, step=42):
    dates = list(pd.Index(unique_dates).sort_values())
    splits = []
    start = train_min
    while start + test_size <= len(dates):
        train_dates = dates[:start]
        test_dates = dates[start:start + test_size]
        splits.append((train_dates, test_dates))
        start += step
    return splits

def safe_auc(y, p):
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, p))

X = model_table[FEATURE_COLUMNS].astype(float)
y = model_table["target"].astype(int)
dates = pd.to_datetime(model_table["date"])

MODELS = {
    "logistic_l2": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000, C=0.8))]),
    "random_forest": RandomForestClassifier(n_estimators=160, max_depth=5, min_samples_leaf=25, random_state=RNG_SEED, n_jobs=-1),
    "gradient_boosting": GradientBoostingClassifier(random_state=RNG_SEED, learning_rate=0.04, n_estimators=130, max_depth=2),
    "mlp": Pipeline([("scale", StandardScaler()), ("model", MLPClassifier(hidden_layer_sizes=(24, 12), alpha=0.02, max_iter=500, random_state=RNG_SEED))]),
}

preds = []
splits = expanding_splits(sorted(dates.unique()), train_min=252, test_size=42, step=42)
for fold, (train_dates, test_dates) in enumerate(splits, 1):
    train_mask = dates.isin(train_dates)
    test_mask = dates.isin(test_dates)
    fold_frame = model_table.loc[test_mask, ["date", "ticker", "target"]].copy()
    for name, model in MODELS.items():
        model.fit(X.loc[train_mask], y.loc[train_mask])
        if hasattr(model, "predict_proba"):
            fold_frame[name] = model.predict_proba(X.loc[test_mask])[:, 1]
        else:
            fold_frame[name] = model.decision_function(X.loc[test_mask])
    fold_frame["ensemble"] = fold_frame[list(MODELS)].mean(axis=1)
    fold_frame["fold"] = fold
    preds.append(fold_frame)

predictions = pd.concat(preds, ignore_index=True)
metrics = {}
for name in list(MODELS) + ["ensemble"]:
    p = predictions[name].astype(float)
    y_true = predictions["target"].astype(int)
    y_hat = (p >= p.median()).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_hat, average="binary", zero_division=0)
    metrics[name] = {
        "roc_auc": safe_auc(y_true, p),
        "brier": float(brier_score_loss(y_true, np.clip(p, 1e-6, 1 - 1e-6))),
        "accuracy_median_threshold": float(accuracy_score(y_true, y_hat)),
        "precision": float(pr),
        "recall": float(rc),
        "f1": float(f1),
    }

# Permutation-style importance on the final training window for the tree model.
final_train = dates < sorted(dates.unique())[-84]
final_test = ~final_train
gbm = MODELS["gradient_boosting"]
gbm.fit(X.loc[final_train], y.loc[final_train])
base = safe_auc(y.loc[final_test], gbm.predict_proba(X.loc[final_test])[:, 1])
importance = {}
for col in FEATURE_COLUMNS:
    Xp = X.loc[final_test].copy()
    Xp[col] = rng.permutation(Xp[col].values)
    importance[col] = float(base - safe_auc(y.loc[final_test], gbm.predict_proba(Xp)[:, 1]))

model_card = {
    "laiqms_modules": ["M", "AI", "Q"],
    "splits": len(splits),
    "features": FEATURE_COLUMNS,
    "metrics": metrics,
    "permutation_importance_auc_drop": importance,
    "leakage_controls": [
        "labels use future returns only after the feature date",
        "walk-forward split uses past dates only",
        "scalers are fit inside each training fold",
        "reported metrics are out-of-sample by fold",
    ],
}
(ARTIFACTS / "s3_model_card.json").write_text(json.dumps(model_card, indent=2))
predictions.to_csv(ARTIFACTS / "s3_walk_forward_predictions.csv", index=False)
print(pd.DataFrame(metrics).T.round(4))


/usr/local/lib/python3.8/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


                   roc_auc   brier  accuracy_median_threshold  precision  \
logistic_l2         0.5070  0.2505                     0.5075     0.5075   
random_forest       0.4829  0.2522                     0.4780     0.4780   
gradient_boosting   0.4884  0.2544                     0.4784     0.4784   
mlp                 0.4830  0.2957                     0.4894     0.4894   
ensemble            0.4829  0.2556                     0.4846     0.4846   

                   recall      f1  
logistic_l2        0.5075  0.5075  
random_forest      0.4780  0.4780  
gradient_boosting  0.4784  0.4784  
mlp                0.4894  0.4894  
ensemble           0.4846  0.4846  


# S4 - LLM Research Layer

Build a local RAG-style research assistant with tokenization, embeddings, attention math, retrieval, and grounded finance checks.


## Mapped LAIQMS Subsections

- L: Tokenization, Embeddings & Vector Geometry
- L: Softmax, Entropy & Cross-Entropy
- L: Attention, Masking & Multi-Head Geometry
- L: Transformer Blocks & Positional Encoding
- L: Evaluation, Calibration & Information Theory
- L: RAG, Inference & Alignment Workflows
- AI: Representation Learning & Embeddings
- AI: Applied AI Evaluation & Calibration
- Q: Financial Instruments & Derivatives


## S4 - LLM Research Layer

**Natural order:** after the model exists, use LLM workflow for research, explanation, and monitoring. The LLM proposes; quant tests verify.

**LAIQMS mapping**
- L / Tokenization: simple financial-token parser.
- L / Embeddings: TF-IDF vector geometry and cosine retrieval.
- L / Softmax/Cross-Entropy: stable softmax and entropy for retrieval confidence.
- L / Attention: scaled dot-product attention from scratch.
- L / RAG: retrieve evidence before writing the research brief.
- AI / Calibration: flag unsupported claims and stale context.
- Q / Finance: every generated statement references data, risk, or market structure.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def stable_softmax(x):
    x = np.asarray(x, dtype=float)
    z = x - np.max(x)
    e = np.exp(z)
    return e / e.sum()

def entropy(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1)
    return float(-(p * np.log(p)).sum())

def scaled_dot_product_attention(Q, K, V, mask=None):
    scores = Q @ K.T / np.sqrt(K.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = np.apply_along_axis(stable_softmax, 1, scores)
    return weights @ V, weights

docs = []
for t in TICKERS:
    last_mom = float(mom_21[t].dropna().iloc[-1])
    last_vol = float(vol_21[t].dropna().iloc[-1])
    last_beta = float(beta_63[t].dropna().iloc[-1])
    docs.append({
        "ticker": t,
        "text": (
            f"{t} latest research note: 21-day momentum {last_mom:.3f}; "
            f"annualized volatility {last_vol:.3f}; beta {last_beta:.3f}; "
            "risk checks include liquidity, sector exposure, drawdown, and transaction costs."
        ),
    })
docs += [
    {"ticker": "SYSTEM", "text": "Point-in-time data is mandatory. Avoid look-ahead bias, survivorship bias, and restated fundamentals."},
    {"ticker": "SYSTEM", "text": "A signal must survive walk-forward validation, transaction costs, turnover constraints, and stress periods."},
    {"ticker": "SYSTEM", "text": "LLM output is useful for parsing and hypothesis generation, but claims must cite retrieved evidence and numerical tests."},
]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
Xdoc = vectorizer.fit_transform([d["text"] for d in docs])

def retrieve(query, k=4):
    q = vectorizer.transform([query])
    sims = cosine_similarity(q, Xdoc).ravel()
    idx = np.argsort(-sims)[:k]
    probs = stable_softmax(sims[idx])
    return [{"score": float(sims[i]), "confidence": float(probs[j]), **docs[i]} for j, i in enumerate(idx)]

query = "Find candidates with momentum, controlled volatility, beta risk, and leak-free validation."
retrieved = retrieve(query, 5)

# Attention demo over retrieved document vectors.
dense = Xdoc[np.argsort(-cosine_similarity(vectorizer.transform([query]), Xdoc).ravel())[:5]].toarray()
Q = dense[:1]
K = dense
V = dense
attended, attn_weights = scaled_dot_product_attention(Q, K, V)

def grounded_brief(ticker):
    evidence = retrieve(f"{ticker} momentum volatility beta risk validation", 4)
    pred_slice = predictions[predictions["ticker"] == ticker].copy()
    score = float(pred_slice["ensemble"].tail(30).mean()) if len(pred_slice) else np.nan
    risk = {
        "vol": float(vol_21[ticker].dropna().iloc[-1]),
        "beta": float(beta_63[ticker].dropna().iloc[-1]),
        "drawdown": float(drawdown_63[ticker].dropna().iloc[-1]),
    }
    unsupported_claims = []
    if score > 0.65 and risk["vol"] > vol_21.stack().quantile(0.85):
        unsupported_claims.append("High model score conflicts with high volatility; require position cap.")
    return {
        "ticker": ticker,
        "model_score_30d_mean": score,
        "risk": risk,
        "evidence": evidence,
        "decision": "candidate" if score >= 0.52 and risk["vol"] < vol_21.stack().quantile(0.75) else "watchlist",
        "unsupported_claims": unsupported_claims,
    }

briefs = [grounded_brief(t) for t in TICKERS]
rag_report = {
    "laiqms_modules": ["L", "AI", "Q"],
    "query": query,
    "retrieval_entropy": entropy([r["confidence"] for r in retrieved]),
    "attention_weights": attn_weights.round(4).tolist(),
    "top_retrieval": retrieved,
    "briefs": briefs,
}
(ARTIFACTS / "s4_rag_research_briefs.json").write_text(json.dumps(rag_report, indent=2))
print(json.dumps({"retrieval_entropy": rag_report["retrieval_entropy"], "top": retrieved[0]}, indent=2))


{
  "retrieval_entropy": 1.6092981128765882,
  "top": {
    "score": 0.18274932820901235,
    "confidence": 0.2066657715322147,
    "ticker": "SYSTEM",
    "text": "A signal must survive walk-forward validation, transaction costs, turnover constraints, and stress periods."
  }
}


# S5 - Portfolio, Execution, Risk

Turn model scores into a constrained portfolio, include costs, run a backtest, stress risk, and define execution controls.


## Mapped LAIQMS Subsections

- Q: Portfolio Theory, Factors & Risk
- Q: Financial Instruments & Derivatives
- Q: Monte Carlo & Numerical Methods
- M: Metrics, Model Selection & Imbalanced Data
- S: Stream Processing, Queues & Backpressure
- S: Applied Trading and ML System Design


## S5 - Portfolio, Execution, Risk

**Natural order:** a model is not a strategy until position sizing, transaction costs, risk limits, and execution logic exist.

**LAIQMS mapping**
- Q / Portfolio Theory: optimizer, constraints, Sharpe, drawdown, beta, VaR, expected shortfall.
- Q / Instruments: option overlay and delta exposure.
- M / Metrics: model score thresholding and out-of-sample interpretation.
- S / Streams/Queues: rebalance queue and order throttling.
- S / Trading System Design: market-data -> signal -> risk -> order path.


In [8]:
def normalize_long_short(scores, long_k=3, short_k=3, gross=1.0):
    scores = scores.dropna().sort_values()
    shorts = scores.head(short_k).index
    longs = scores.tail(long_k).index
    w = pd.Series(0.0, index=scores.index)
    if len(longs):
        w.loc[longs] = gross / 2 / len(longs)
    if len(shorts):
        w.loc[shorts] = -gross / 2 / len(shorts)
    return w

pred_wide = predictions.pivot_table(index="date", columns="ticker", values="ensemble", aggfunc="last").sort_index()
ret = DATA["returns"].reindex(pred_wide.index)
rebal_dates = pred_wide.index[::21]
weights_by_day = pd.DataFrame(0.0, index=pred_wide.index, columns=TICKERS)
current = pd.Series(0.0, index=TICKERS)
rebalance_queue = deque()

for d in pred_wide.index:
    if d in set(rebal_dates):
        target = normalize_long_short(pred_wide.loc[d], 3, 3, gross=1.0).reindex(TICKERS).fillna(0.0)
        rebalance_queue.append({"date": str(d.date()), "orders": (target - current).round(5).to_dict()})
        current = target
    weights_by_day.loc[d] = current

strategy_ret_gross = (weights_by_day.shift().fillna(0.0) * ret).sum(axis=1)
turnover = weights_by_day.diff().abs().sum(axis=1).fillna(0.0)
cost_bps = 6
strategy_ret = strategy_ret_gross - turnover * cost_bps / 10000
wealth = (1 + strategy_ret).cumprod()
bench = ret.mean(axis=1).reindex(strategy_ret.index)

beta = float(np.cov(strategy_ret.dropna(), bench.loc[strategy_ret.dropna().index].dropna())[0, 1] / np.var(bench.dropna()))
alpha_daily = float(strategy_ret.mean() - beta * bench.mean())
stress = {}
for name, shock in {"mild_liquidity_cost": 0.0005, "severe_liquidity_cost": 0.0015, "crisis_gap": 0.006}.items():
    stressed = strategy_ret - turnover * shock
    if name == "crisis_gap":
        stressed = stressed - (DATA["regime"].reindex(stressed.index).fillna(0).eq(2).astype(float) * shock)
    stress[name] = {
        "ann_return": float(stressed.mean() * 252),
        "ann_vol": float(stressed.std() * np.sqrt(252)),
        "max_drawdown": max_drawdown((1 + stressed).cumprod()),
    }

# Simple option overlay example: delta hedge one long call exposure on the top candidate.
latest_scores = pred_wide.iloc[-1].sort_values()
top_name = latest_scores.index[-1]
S0 = float(prices[top_name].iloc[-1])
sig = float(vol_21[top_name].dropna().iloc[-1])
call = black_scholes_call(S0, S0 * 1.03, 45 / 365, RISK_FREE, sig)
delta_hedge_shares = -call["delta"]

risk_report = {
    "laiqms_modules": ["Q", "M", "S"],
    "ann_return": float(strategy_ret.mean() * 252),
    "ann_vol": float(strategy_ret.std() * np.sqrt(252)),
    "sharpe": float((strategy_ret.mean() * 252 - RISK_FREE) / (strategy_ret.std() * np.sqrt(252))),
    "max_drawdown": max_drawdown(wealth),
    "beta_to_equal_weight": beta,
    "alpha_daily": alpha_daily,
    "daily_var_95": float(np.quantile(strategy_ret.dropna(), 0.05)),
    "daily_expected_shortfall_95": float(strategy_ret[strategy_ret <= np.quantile(strategy_ret.dropna(), 0.05)].mean()),
    "average_turnover": float(turnover.mean()),
    "option_overlay": {"ticker": top_name, "call": call, "delta_hedge_shares_per_call": float(delta_hedge_shares)},
    "stress": stress,
    "latest_rebalance_order": rebalance_queue[-1] if rebalance_queue else {},
    "execution_controls": [
        "position cap by volatility bucket",
        "turnover throttle",
        "liquidity/cost gate before order creation",
        "model-score stale check",
        "kill switch when drawdown or data freshness breaches limits",
    ],
}
(ARTIFACTS / "s5_strategy_risk_report.json").write_text(json.dumps(risk_report, indent=2))
print(pd.Series({k: risk_report[k] for k in ["ann_return", "ann_vol", "sharpe", "max_drawdown", "average_turnover"]}).round(4))


ann_return          0.0164
ann_vol             0.1154
sharpe             -0.2301
max_drawdown       -0.1433
average_turnover    0.0591
dtype: float64


# S6 - AWS, System Design, Final Defense

Package the capstone as a production architecture with AWS services, monitoring, security, cost controls, and interview-defense prompts.


## Mapped LAIQMS Subsections

- S: System Design Fundamentals & Estimation
- S: Online Processing Systems
- S: Batch Processing & Storage Systems
- S: Stream Processing, Queues & Backpressure
- S: Distributed Consistency & Fault Tolerance
- S: Applied Trading and ML System Design
- S: AWS Cloud Concepts & Economics
- S: AWS Shared Responsibility, IAM & Security
- S: AWS Compute & Serverless
- S: AWS Storage & Databases
- S: AWS Networking & Content Delivery
- S: AWS Monitoring, Reliability & Operations
- S: AWS Billing, Pricing & Support


## S6 - AWS, System Design, Final Defense

**Natural order:** once the strategy works as a notebook, design the production path.

**LAIQMS mapping**
- S / System Design: ingestion, batch jobs, online APIs, queues, failure modes.
- AWS / Cloud Practitioner: IAM, S3, Lambda, ECS, DynamoDB, CloudWatch, EventBridge, VPC, CloudFront, budgets.
- L / AI / M / Q: the deployed system must monitor model drift, LLM grounding, data freshness, and portfolio risk.


In [9]:
architecture = {
    "name": "laiqms-alpha-research-and-risk-system",
    "data_layer": {
        "raw_market_data": "s3://mafn-capstone/raw/market/",
        "feature_store": "s3://mafn-capstone/curated/features/",
        "model_registry": "s3://mafn-capstone/models/",
        "research_corpus": "s3://mafn-capstone/rag/documents/",
    },
    "compute": {
        "batch_feature_job": "AWS Batch or ECS scheduled task",
        "model_training": "SageMaker training job or ECS task",
        "inference_api": "Lambda/API Gateway for low-volume class demo; ECS/Fargate for heavier use",
        "notebook": "SageMaker Studio or local Jupyter template",
    },
    "state": {
        "predictions": "DynamoDB table keyed by date#ticker",
        "risk_reports": "S3 JSON artifacts and Athena external table",
        "release_toggles": "S3 JSON release state behind authenticated API",
    },
    "networking": {
        "cdn": "CloudFront for student downloads",
        "private_subnets": "model jobs and databases",
        "public_edge": "API Gateway + WAF",
    },
    "security": {
        "iam": "least-privilege role per job",
        "encryption": "SSE-S3 or KMS for S3; DynamoDB encryption at rest",
        "auth": "Cognito group mafn-admins controls release",
        "audit": "CloudTrail + application audit log",
    },
    "monitoring": {
        "data_freshness": "CloudWatch metric: minutes since last market-data update",
        "model_drift": "population stability index / feature distribution shift",
        "llm_grounding": "retrieval evidence count and unsupported-claim rate",
        "risk": "drawdown, VaR, turnover, beta, exposure caps",
    },
}

iam_policy_example = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject"], "Resource": ["arn:aws:s3:::mafn-capstone/*"]},
        {"Effect": "Allow", "Action": ["dynamodb:GetItem", "dynamodb:PutItem", "dynamodb:Query"], "Resource": ["arn:aws:dynamodb:us-east-1:123456789012:table/mafn-capstone-*"]},
        {"Effect": "Allow", "Action": ["cloudwatch:PutMetricData"], "Resource": "*"},
    ],
}

step_functions_skeleton = {
    "Comment": "LAIQMS capstone daily pipeline",
    "StartAt": "IngestMarketData",
    "States": {
        "IngestMarketData": {"Type": "Task", "Resource": "arn:aws:lambda:REGION:ACCT:function:ingest-market-data", "Next": "BuildFeatures"},
        "BuildFeatures": {"Type": "Task", "Resource": "arn:aws:states:::batch:submitJob.sync", "Next": "ScoreModel"},
        "ScoreModel": {"Type": "Task", "Resource": "arn:aws:lambda:REGION:ACCT:function:score-model", "Next": "RiskChecks"},
        "RiskChecks": {"Type": "Task", "Resource": "arn:aws:lambda:REGION:ACCT:function:risk-checks", "Next": "PublishReports"},
        "PublishReports": {"Type": "Task", "Resource": "arn:aws:lambda:REGION:ACCT:function:publish-reports", "End": True},
    },
}

monthly_cost_estimate = pd.DataFrame([
    {"service": "S3", "assumption": "50 GB artifacts/data", "monthly_usd": 1.25},
    {"service": "Lambda", "assumption": "100k requests", "monthly_usd": 2.00},
    {"service": "DynamoDB", "assumption": "on-demand class demo", "monthly_usd": 5.00},
    {"service": "CloudWatch", "assumption": "metrics/logs/alarms", "monthly_usd": 8.00},
    {"service": "ECS/Batch", "assumption": "scheduled training jobs", "monthly_usd": 30.00},
    {"service": "CloudFront", "assumption": "notebook downloads", "monthly_usd": 3.00},
])

runbook = [
    "If data freshness > threshold: stop scoring, alert instructor, publish stale-data banner.",
    "If model drift breaches threshold: keep old model, require admin review before release.",
    "If unsupported LLM claim rate rises: disable research assistant output for students.",
    "If drawdown or turnover limit breaches: set strategy status to review-only.",
    "If cost budget forecast exceeds limit: reduce training frequency and cap artifact retention.",
]

defense_prompts = [
    "Explain why the model is not allowed to train on future data.",
    "Defend the portfolio optimizer against noisy expected returns.",
    "Explain where the LLM is useful and where it must be constrained.",
    "Draw the AWS production architecture and name the failure modes.",
    "Describe the monitoring that would stop this strategy from going live.",
]

s6 = {
    "laiqms_modules": ["S", "AWS", "L", "AI", "Q", "M"],
    "architecture": architecture,
    "iam_policy_example": iam_policy_example,
    "step_functions_skeleton": step_functions_skeleton,
    "monthly_cost_estimate": monthly_cost_estimate.to_dict(orient="records"),
    "runbook": runbook,
    "defense_prompts": defense_prompts,
}
(ARTIFACTS / "s6_aws_architecture.json").write_text(json.dumps(s6, indent=2))
(ARTIFACTS / "s6_final_defense_prompts.md").write_text("\n".join(f"- {x}" for x in defense_prompts))
print(monthly_cost_estimate)


      service               assumption  monthly_usd
0          S3     50 GB artifacts/data         1.25
1      Lambda            100k requests         2.00
2    DynamoDB     on-demand class demo         5.00
3  CloudWatch      metrics/logs/alarms         8.00
4   ECS/Batch  scheduled training jobs        30.00
5  CloudFront       notebook downloads         3.00
